# E2: Episodic Trace Memory on ALFWorld

Validates the full episodic trace memory pipeline:

1. **Ingest** expert trajectories as multi-step episodes
2. **Retrieve** analogous episodes by initial-state similarity + reward filter
3. **Inject** retrieved trajectories as procedural context
4. **Evaluate** plan quality against expert trajectories

**Dataset**: AgentInstruct ALFWorld split (336 GPT-4 expert trajectories)

### Conditions

| Condition | Description |
|-----------|-------------|
| **NoMemory** | Zero-shot agent |
| **RandomTrajectory** | k random expert trajectories |
| **FlatCosine** | k most similar by numpy cosine (no CVX) |
| **CVX-Episodic** | k most similar via CVX temporal search |

### Metrics (all computed against expert trajectory)

| Metric | Description |
|--------|-------------|
| **Action Overlap** | Fraction of expert action verbs present in generated plan |
| **Semantic Similarity** | Cosine similarity between plan and expert trajectory embeddings |
| **Step Ratio** | generated_steps / expert_steps (closer to 1.0 = better) |
| **Object Recall** | Fraction of expert-mentioned objects appearing in the plan |

### Experimental Design

- **Leave-one-out**: Each episode evaluated with itself excluded from retrieval
- **n=100** episodes (stratified across task types)
- **Breakdown** by task type (put, clean, cool, heat, examine)

In [1]:
import os, json, time, re
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from scipy import stats

# === MODEL CONFIG ===
USE_OLLAMA = True
OLLAMA_MODEL = 'qwen2.5-coder:7b-instruct'
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
    LLM_MODEL = OLLAMA_MODEL
else:
    assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY'
    client = OpenAI()
    LLM_MODEL = 'gpt-4o-mini'

EMBED_MODEL = 'all-MiniLM-L6-v2'
TOP_K = 3
N_EVAL = 100  # episodes to evaluate

DATA_DIR = Path('../data/episodic')
DATA_DIR.mkdir(parents=True, exist_ok=True)

embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'Embedding: {EMBED_MODEL} (D={D})')
print(f'LLM: {LLM_MODEL} ({"Ollama" if USE_OLLAMA else "OpenAI"})')
print(f'Evaluation: n={N_EVAL} episodes, top-k={TOP_K}')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11033.70it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding: all-MiniLM-L6-v2 (D=384)
LLM: qwen2.5-coder:7b-instruct (Ollama)
Evaluation: n=100 episodes, top-k=3


## 1. Load AgentInstruct Trajectories

Expert ALFWorld trajectories generated by GPT-4.

In [2]:
from datasets import load_dataset
import re

# AgentInstruct: expert ALFWorld trajectories (GPT-4)
raw_ds = load_dataset('zai-org/agentinstruct', split='alfworld')
print(f'AgentInstruct alfworld: {len(raw_ds)} trajectories')

# Parse conversation format into structured episodes
# These are GPT-4 expert trajectories — all are successful demonstrations (reward=1.0)
ds = []
for row in raw_ds:
    convs = row['conversations']
    
    # Extract task from the human turn that describes the environment
    task = ''
    steps = []
    for turn in convs:
        text = turn['value']
        # Task is in the turn containing "Your task is to:"
        if 'Your task is to:' in text:
            match = re.search(r'Your task is to:\s*(.+)', text)
            if match:
                task = match.group(1).strip()
        # GPT actions are the agent steps
        elif turn['from'] == 'gpt' and ('ACTION:' in text or 'THOUGHT:' in text):
            action_match = re.search(r'ACTION:\s*(.+)', text)
            action = action_match.group(1).strip() if action_match else ''
            thought_match = re.search(r'THOUGHT:\s*(.+?)(?:\n|ACTION:)', text, re.DOTALL)
            thought = thought_match.group(1).strip() if thought_match else ''
            if action:
                steps.append({'action': action, 'observation': thought})
        # Human feedback (environment response) — attach to last step
        elif turn['from'] == 'human' and len(steps) > 0 and 'Your task is to:' not in text:
            steps[-1]['observation'] = text[:200]
    
    if task and steps:
        # AgentInstruct trajectories are expert demonstrations — all successful
        reward = 1.0
        
        # Infer task type from task description
        task_lower = task.lower()
        if 'clean' in task_lower: task_type = 'clean'
        elif 'heat' in task_lower: task_type = 'heat'
        elif 'cool' in task_lower: task_type = 'cool'
        elif 'examine' in task_lower or 'look' in task_lower: task_type = 'examine'
        elif 'put' in task_lower or 'find' in task_lower: task_type = 'put'
        else: task_type = 'pick'
        
        ds.append({
            'task': task,
            'steps': steps,
            'reward': reward,
            'task_type': task_type,
        })

print(f'Parsed {len(ds)} episodes from AgentInstruct')
n_success = sum(1 for e in ds if e['reward'] >= 0.7)
print(f'  Successful: {n_success}/{len(ds)} (expert demonstrations)')
print(f'  Task types: {dict(pd.Series([e["task_type"] for e in ds]).value_counts())}')
print(f'  Steps/episode: min={min(len(e["steps"]) for e in ds)}, '
      f'median={np.median([len(e["steps"]) for e in ds]):.0f}, '
      f'max={max(len(e["steps"]) for e in ds)}')
print(f'\nSample task: {ds[0]["task"][:80]}')
print(f'Sample steps ({len(ds[0]["steps"])}):')
for s in ds[0]['steps'][:3]:
    print(f'  Action: {s["action"][:60]}')
    print(f'  Obs: {s["observation"][:60]}')

AgentInstruct alfworld: 336 trajectories
Parsed 336 episodes from AgentInstruct
  Successful: 336/336 (expert demonstrations)
  Task types: {'put': np.int64(158), 'clean': np.int64(68), 'cool': np.int64(49), 'examine': np.int64(37), 'heat': np.int64(24)}
  Steps/episode: min=3, median=10, max=35

Sample task: find two laptop and put them in bed.
Sample steps (14):
  Action: go to diningtable 1
  Obs: On the diningtable 1, you see a alarmclock 2, a bowl 2, a cd
  Action: take laptop 1 from diningtable 1
  Obs: You pick up the laptop 1 from the diningtable 1.
  Action: go to bed 1
  Obs: On the bed 1, you see a pillow 2, and a pillow 1.


## 2. Build Episodic Index

In [3]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'alfworld_episodic.cvx')
FLAT_EMB_PATH = str(DATA_DIR / 'alfworld_task_embeddings.npy')

def encode_entity(episode_id, step_index):
    return (episode_id << 16) | step_index

if os.path.exists(INDEX_PATH):
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded: {len(index)} points')
else:
    print('Building episodic memory...')
    index = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=50)
    
    episode_data = {}
    
    for ep_idx, traj in enumerate(ds):
        task = traj['task']
        steps = traj['steps']
        reward = traj['reward']
        
        for s_idx, step in enumerate(steps):
            text = f"Task: {task} | Action: {step.get('action', '')} | Obs: {step.get('observation', '')}"
            vec = embedder.encode(text[:500]).tolist()
            eid = encode_entity(ep_idx, s_idx)
            index.insert(entity_id=eid, timestamp=ep_idx * 10000 + s_idx, vector=vec)
        
        episode_data[ep_idx] = {
            'task': task,
            'steps': steps,
            'reward': reward,
            'n_steps': len(steps),
            'task_type': traj.get('task_type', 'unknown'),
        }
    
    index.save(INDEX_PATH)
    with open(str(DATA_DIR / 'alfworld_metadata.json'), 'w') as f:
        json.dump(episode_data, f)
    print(f'Built: {len(index)} points ({len(episode_data)} episodes)')

with open(str(DATA_DIR / 'alfworld_metadata.json')) as f:
    episode_data = {int(k): v for k, v in json.load(f).items()}

# Build flat embedding matrix for cosine baseline (task descriptions only)
if os.path.exists(FLAT_EMB_PATH):
    task_embeddings = np.load(FLAT_EMB_PATH)
    print(f'Loaded flat task embeddings: {task_embeddings.shape}')
else:
    print('Building flat task embeddings...')
    task_embeddings = embedder.encode([episode_data[i]['task'] for i in range(len(episode_data))])
    np.save(FLAT_EMB_PATH, task_embeddings)
    print(f'Built flat embeddings: {task_embeddings.shape}')

n_success = sum(1 for e in episode_data.values() if e['reward'] >= 0.7)
print(f'{len(episode_data)} episodes ({n_success} successful), flat matrix {task_embeddings.shape}')

Building episodic memory...


Built: 4542 points (336 episodes)
Building flat task embeddings...


Built flat embeddings: (336, 384)
336 episodes (336 successful), flat matrix (336, 384)


## 3. Retrieval & Evaluation

In [4]:
def retrieve_cvx(task_description, top_k=TOP_K, min_reward=0.7, exclude_ep=None):
    """Retrieve similar episodes via CVX temporal search."""
    query_vec = embedder.encode(f'Task: {task_description}').tolist()
    results = index.search(vector=query_vec, k=top_k * 20)
    
    seen = set()
    episodes = []
    for node_id, ts, score in results:
        ep_idx = ts // 10000
        if ep_idx in seen or ep_idx not in episode_data:
            continue
        if exclude_ep is not None and ep_idx == exclude_ep:
            continue
        ep = episode_data[ep_idx]
        if ep['reward'] < min_reward:
            continue
        seen.add(ep_idx)
        episodes.append({'episode_id': ep_idx, 'similarity': score, **ep})
        if len(episodes) >= top_k:
            break
    return episodes


def retrieve_flat_cosine(task_description, top_k=TOP_K, exclude_ep=None):
    """Retrieve similar episodes via flat numpy cosine similarity."""
    query_vec = embedder.encode(task_description)
    query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-8)
    emb_norms = task_embeddings / (np.linalg.norm(task_embeddings, axis=1, keepdims=True) + 1e-8)
    sims = emb_norms @ query_norm
    
    top_indices = np.argsort(sims)[::-1]
    episodes = []
    for idx in top_indices:
        idx = int(idx)
        if exclude_ep is not None and idx == exclude_ep:
            continue
        if idx in episode_data:
            episodes.append({'episode_id': idx, 'similarity': float(sims[idx]), **episode_data[idx]})
        if len(episodes) >= top_k:
            break
    return episodes


def retrieve_random(top_k=TOP_K, exclude_ep=None):
    """Random baseline."""
    candidates = [i for i in range(len(episode_data)) if i != exclude_ep]
    indices = np.random.choice(candidates, size=top_k, replace=False)
    return [{'episode_id': int(i), **episode_data[int(i)]} for i in indices]


def format_trajectory(episodes):
    """Format episodes for LLM prompt."""
    lines = []
    for i, ep in enumerate(episodes, 1):
        lines.append(f'[Past Episode {i} — Success, {ep["n_steps"]} steps]')
        lines.append(f'Task: {ep["task"]}')
        for j, step in enumerate(ep['steps'][:12], 1):
            if isinstance(step, dict):
                lines.append(f'  Step {j}: {step.get("action", "")} → {step.get("observation", "")[:80]}')
            else:
                lines.append(f'  Step {j}: {str(step)[:100]}')
        lines.append('')
    return '\n'.join(lines)


# === EVALUATION METRICS ===

# ALFWorld action vocabulary
ACTION_VERBS = {'go', 'take', 'put', 'open', 'close', 'use', 'examine', 'look', 'clean', 'heat', 'cool', 'toggle'}

def extract_actions(text):
    """Extract action verbs from a plan or trajectory."""
    words = set(text.lower().split())
    return words & ACTION_VERBS

def extract_objects(text):
    """Extract ALFWorld objects (noun + number patterns like 'apple 1', 'fridge 1')."""
    return set(re.findall(r'[a-z]+(?:table|shelf|counter|fridge|microwave|sink|toilet|drawer|cabinet|desk|bed|sofa|armchair|lamp|stove)\s*\d*', text.lower()))

def compute_plan_metrics(plan_text, expert_episode):
    """Compute multiple quality metrics between generated plan and expert trajectory."""
    expert_text = ' '.join(
        step.get('action', '') + ' ' + step.get('observation', '')
        for step in expert_episode['steps']
        if isinstance(step, dict)
    )
    
    # 1. Step ratio (closer to 1.0 = better)
    plan_steps = len([l for l in plan_text.strip().split('\n') if l.strip()])
    step_ratio = plan_steps / max(expert_episode['n_steps'], 1)
    
    # 2. Action verb overlap
    plan_actions = extract_actions(plan_text)
    expert_actions = extract_actions(expert_text)
    action_overlap = len(plan_actions & expert_actions) / max(len(expert_actions), 1)
    
    # 3. Object recall
    plan_objects = extract_objects(plan_text)
    expert_objects = extract_objects(expert_text)
    object_recall = len(plan_objects & expert_objects) / max(len(expert_objects), 1)
    
    # 4. Semantic similarity (plan embedding vs expert trajectory embedding)
    plan_emb = embedder.encode(plan_text[:500])
    expert_emb = embedder.encode(expert_text[:500])
    cos_sim = float(np.dot(plan_emb, expert_emb) / (np.linalg.norm(plan_emb) * np.linalg.norm(expert_emb) + 1e-8))
    
    return {
        'n_steps': plan_steps,
        'step_ratio': step_ratio,
        'action_overlap': action_overlap,
        'object_recall': object_recall,
        'semantic_sim': cos_sim,
    }


# Test
test_task = episode_data[0]['task']
print(f'Query: {test_task}\n')
for name, fn in [('CVX', lambda: retrieve_cvx(test_task, exclude_ep=0)),
                 ('Flat', lambda: retrieve_flat_cosine(test_task, exclude_ep=0))]:
    eps = fn()
    print(f'{name}: {len(eps)} episodes')
    for ep in eps:
        print(f'  [{ep["episode_id"]}] sim={ep["similarity"]:.3f}: {ep["task"][:50]}')
    print()

Query: find two laptop and put them in bed.

CVX: 3 episodes
  [259] sim=0.307: put two laptop in bed.
  [45] sim=0.493: find two book and put them in bed.
  [86] sim=0.520: put two book in bed.

Flat: 3 episodes
  [259] sim=0.943: put two laptop in bed.
  [45] sim=0.673: find two book and put them in bed.
  [55] sim=0.654: put two book in bed.



In [5]:
def solve_with_llm(task, context=''):
    """Ask LLM to produce a plan for the task."""
    system = (
        'You are an embodied agent in a household. Given a task, produce a step-by-step '
        'action plan. Each step should be one action like "go to countertop 1", "take apple 1 from countertop 1", '
        '"put apple 1 in/on fridge 1". Use the exact ALFWorld action format. '
        'Output ONLY the numbered steps, no explanation.'
    )
    user_parts = []
    if context:
        user_parts.append('Here are examples of how similar tasks were solved:\n')
        user_parts.append(context)
        user_parts.append('\n---\nNow solve this task:\n')
    user_parts.append(f'Task: {task}')
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=0.0,
        max_tokens=512,
    )
    return response.choices[0].message.content


# Stratified sample: proportional to task types
task_type_counts = pd.Series([episode_data[i]['task_type'] for i in range(len(episode_data))]).value_counts()
eval_indices = []
np.random.seed(42)
for tt, count in task_type_counts.items():
    tt_indices = [i for i in range(len(episode_data)) if episode_data[i]['task_type'] == tt]
    n_sample = max(1, int(N_EVAL * count / len(episode_data)))
    eval_indices.extend(np.random.choice(tt_indices, size=min(n_sample, len(tt_indices)), replace=False))
eval_indices = eval_indices[:N_EVAL]
print(f'Evaluation: {len(eval_indices)} episodes (stratified)')
print(f'  Types: {dict(pd.Series([episode_data[i]["task_type"] for i in eval_indices]).value_counts())}')

# Define conditions (with leave-one-out)
def get_context_nomemory(task, ep_idx, k):
    return ''

def get_context_random(task, ep_idx, k):
    return format_trajectory(retrieve_random(top_k=k, exclude_ep=ep_idx))

def get_context_flat(task, ep_idx, k):
    return format_trajectory(retrieve_flat_cosine(task, top_k=k, exclude_ep=ep_idx))

def get_context_cvx(task, ep_idx, k):
    return format_trajectory(retrieve_cvx(task, top_k=k, exclude_ep=ep_idx))

conditions = {
    'NoMemory': get_context_nomemory,
    'RandomTrajectory': get_context_random,
    'FlatCosine': get_context_flat,
    'CVX-Episodic': get_context_cvx,
}

# Run evaluation
all_results = {cond: [] for cond in conditions}
t0 = time.perf_counter()

for i, ep_idx in enumerate(eval_indices):
    ep = episode_data[ep_idx]
    task = ep['task']
    
    for cond_name, get_ctx in conditions.items():
        context = get_ctx(task, ep_idx, TOP_K)
        plan = solve_with_llm(task, context)
        metrics = compute_plan_metrics(plan, ep)
        metrics['task_type'] = ep['task_type']
        metrics['ep_idx'] = ep_idx
        all_results[cond_name].append(metrics)
    
    if (i + 1) % 20 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (i + 1) * (len(eval_indices) - i - 1)
        print(f'[{i+1}/{len(eval_indices)}] elapsed={elapsed:.0f}s, ETA={eta:.0f}s')

print(f'\nDone in {time.perf_counter() - t0:.0f}s')

# Aggregate results
print('\n=== OVERALL RESULTS ===')
print(f'{"Condition":>18} | {"Steps":>6} | {"StepRatio":>9} | {"ActionOvl":>9} | {"ObjRecall":>9} | {"SemanticSim":>11}')
print('-' * 75)
for cond, res in all_results.items():
    df_r = pd.DataFrame(res)
    print(f'{cond:>18} | {df_r["n_steps"].mean():>6.1f} | {df_r["step_ratio"].mean():>9.2f} | '
          f'{df_r["action_overlap"].mean():>9.2f} | {df_r["object_recall"].mean():>9.2f} | '
          f'{df_r["semantic_sim"].mean():>11.3f}')

Evaluation: 99 episodes (stratified)
  Types: {'put': np.int64(47), 'clean': np.int64(20), 'cool': np.int64(14), 'examine': np.int64(11), 'heat': np.int64(7)}


[20/99] elapsed=150s, ETA=592s


[40/99] elapsed=291s, ETA=429s


[60/99] elapsed=439s, ETA=285s


[80/99] elapsed=590s, ETA=140s



Done in 695s

=== OVERALL RESULTS ===
         Condition |  Steps | StepRatio | ActionOvl | ObjRecall | SemanticSim
---------------------------------------------------------------------------
          NoMemory |    6.0 |      0.56 |      0.60 |      0.10 |       0.600
  RandomTrajectory |    7.6 |      0.72 |      0.78 |      0.17 |       0.612
        FlatCosine |    8.3 |      0.74 |      0.88 |      0.30 |       0.700
      CVX-Episodic |    8.4 |      0.74 |      0.88 |      0.28 |       0.708


In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = {'NoMemory': '#95a5a6', 'RandomTrajectory': '#3498db', 'FlatCosine': '#f39c12', 'CVX-Episodic': '#e74c3c'}

# Multi-metric bar chart
metrics_to_plot = ['semantic_sim', 'action_overlap', 'object_recall', 'step_ratio']
metric_labels = ['Semantic Similarity', 'Action Verb Overlap', 'Object Recall', 'Step Ratio']

fig = make_subplots(rows=2, cols=2, subplot_titles=metric_labels)

for idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    row, col = idx // 2 + 1, idx % 2 + 1
    for cond in conditions:
        vals = [r[metric] for r in all_results[cond]]
        fig.add_trace(go.Bar(
            x=[cond], y=[np.mean(vals)],
            error_y=dict(type='data', array=[np.std(vals)], visible=True),
            marker_color=colors[cond],
            name=cond, showlegend=(idx == 0),
        ), row=row, col=col)

fig.update_layout(height=600, width=900, title=f'Plan Quality Metrics (n={len(eval_indices)}, k={TOP_K})')
fig.show()

# Task-type breakdown for semantic similarity (key metric)
fig2 = go.Figure()
task_types = sorted(set(r['task_type'] for r in all_results['NoMemory']))

for cond in conditions:
    means = []
    for tt in task_types:
        vals = [r['semantic_sim'] for r in all_results[cond] if r['task_type'] == tt]
        means.append(np.mean(vals) if vals else 0)
    fig2.add_trace(go.Bar(x=task_types, y=means, name=cond, marker_color=colors[cond]))

fig2.update_layout(
    title='Semantic Similarity by Task Type',
    xaxis_title='Task Type', yaxis_title='Cosine Similarity',
    barmode='group', height=400, width=800,
)
fig2.show()

## 4. Statistical Tests

In [7]:
# Paired Wilcoxon signed-rank tests on semantic similarity (primary metric)
print('=== Wilcoxon Signed-Rank Tests (semantic similarity) ===\n')

pairs = [
    ('CVX-Episodic', 'NoMemory'),
    ('CVX-Episodic', 'FlatCosine'),
    ('CVX-Episodic', 'RandomTrajectory'),
    ('FlatCosine', 'NoMemory'),
    ('FlatCosine', 'RandomTrajectory'),
]

for a, b in pairs:
    a_vals = np.array([r['semantic_sim'] for r in all_results[a]])
    b_vals = np.array([r['semantic_sim'] for r in all_results[b]])
    diff = a_vals - b_vals
    
    stat, p_val = stats.wilcoxon(diff, alternative='greater')
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    mean_diff = np.mean(diff)
    print(f'{a} vs {b}: Δ={mean_diff:+.4f}, W={stat:.0f}, p={p_val:.4f} {sig}')

# Same for object recall
print('\n=== Wilcoxon Signed-Rank Tests (object recall) ===\n')
for a, b in pairs:
    a_vals = np.array([r['object_recall'] for r in all_results[a]])
    b_vals = np.array([r['object_recall'] for r in all_results[b]])
    diff = a_vals - b_vals
    
    # Handle zero diffs
    nonzero = diff[diff != 0]
    if len(nonzero) > 0:
        stat, p_val = stats.wilcoxon(nonzero, alternative='greater')
    else:
        stat, p_val = 0, 1.0
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    mean_diff = np.mean(diff)
    print(f'{a} vs {b}: Δ={mean_diff:+.4f}, W={stat:.0f}, p={p_val:.4f} {sig}')

# Retrieval overlap analysis
print('\n=== Retrieval Overlap: CVX vs FlatCosine ===')
overlap_counts = []
for ep_idx in eval_indices:
    task = episode_data[ep_idx]['task']
    cvx_eps = retrieve_cvx(task, top_k=TOP_K, exclude_ep=ep_idx)
    flat_eps = retrieve_flat_cosine(task, top_k=TOP_K, exclude_ep=ep_idx)
    cvx_ids = {ep['episode_id'] for ep in cvx_eps}
    flat_ids = {ep['episode_id'] for ep in flat_eps}
    overlap = len(cvx_ids & flat_ids) / TOP_K if TOP_K > 0 else 0
    overlap_counts.append(overlap)

print(f'Mean overlap: {np.mean(overlap_counts):.1%}')
print(f'Full overlap: {sum(1 for o in overlap_counts if o == 1.0)} / {len(eval_indices)}')
print(f'Zero overlap: {sum(1 for o in overlap_counts if o == 0.0)} / {len(eval_indices)}')

# Summary table
print('\n=== FINAL SUMMARY ===')
print(f'{"Condition":>18} | {"Sem.Sim":>8} | {"ActionOvl":>9} | {"ObjRecall":>9} | {"StepRatio":>9}')
print('-' * 62)
for cond in conditions:
    df_r = pd.DataFrame(all_results[cond])
    print(f'{cond:>18} | {df_r["semantic_sim"].mean():>7.3f}± | {df_r["action_overlap"].mean():>8.2f}± | '
          f'{df_r["object_recall"].mean():>8.2f}± | {df_r["step_ratio"].mean():>8.2f}±')

print(f'\nModel: {LLM_MODEL}')
print(f'Dataset: AgentInstruct ALFWorld ({len(episode_data)} episodes)')
print(f'Evaluated: {len(eval_indices)} episodes (leave-one-out), k={TOP_K}')
print('CVX features: TemporalIndex, insert, search, save/load, episode encoding, reward filter')

=== Wilcoxon Signed-Rank Tests (semantic similarity) ===

CVX-Episodic vs NoMemory: Δ=+0.1081, W=4521, p=0.0000 ***
CVX-Episodic vs FlatCosine: Δ=+0.0079, W=1071, p=0.2538 ns
CVX-Episodic vs RandomTrajectory: Δ=+0.0961, W=4035, p=0.0000 ***
FlatCosine vs NoMemory: Δ=+0.1002, W=4389, p=0.0000 ***
FlatCosine vs RandomTrajectory: Δ=+0.0882, W=3968, p=0.0000 ***

=== Wilcoxon Signed-Rank Tests (object recall) ===

CVX-Episodic vs NoMemory: Δ=+0.1869, W=332, p=0.0000 ***
CVX-Episodic vs FlatCosine: Δ=-0.0160, W=4, p=0.8438 ns
CVX-Episodic vs RandomTrajectory: Δ=+0.1103, W=162, p=0.0029 **
FlatCosine vs NoMemory: Δ=+0.2029, W=385, p=0.0000 ***
FlatCosine vs RandomTrajectory: Δ=+0.1263, W=158, p=0.0006 ***

=== Retrieval Overlap: CVX vs FlatCosine ===


Mean overlap: 67.0%
Full overlap: 25 / 99
Zero overlap: 4 / 99

=== FINAL SUMMARY ===
         Condition |  Sem.Sim | ActionOvl | ObjRecall | StepRatio
--------------------------------------------------------------
          NoMemory |   0.600± |     0.60± |     0.10± |     0.56±
  RandomTrajectory |   0.612± |     0.78± |     0.17± |     0.72±
        FlatCosine |   0.700± |     0.88± |     0.30± |     0.74±
      CVX-Episodic |   0.708± |     0.88± |     0.28± |     0.74±

Model: qwen2.5-coder:7b-instruct
Dataset: AgentInstruct ALFWorld (336 episodes)
Evaluated: 99 episodes (leave-one-out), k=3
CVX features: TemporalIndex, insert, search, save/load, episode encoding, reward filter
